In [1]:
import pandas as pd
import numpy as np

In [5]:
ieee = pd.read_csv(
    "ieee_data/train_transaction.csv",
    usecols=["TransactionDT", "isFraud"]
)

print("Shape:", ieee.shape)

print("\nClass distribution:")
print(ieee["isFraud"].value_counts())

print("\nTime statistics by class:")
print(
    ieee.groupby("isFraud")["TransactionDT"]
    .agg(["count", "min", "median", "mean", "max"])
)

Shape: (590540, 2)

Class distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64

Time statistics by class:
          count    min     median          mean       max
isFraud                                                  
0        569877  86400  7271678.0  7.360791e+06  15811131
1         20663  89760  7575230.0  7.690033e+06  15810876


In [6]:
# Divide the full IEEE-CIS timeline into 10 temporal bins
ieee["TimeBin"] = pd.qcut(
    ieee["TransactionDT"],
    q=10,
    duplicates="drop"
)

time_distribution = pd.crosstab(
    ieee["TimeBin"],
    ieee["isFraud"]
)

print(time_distribution)

isFraud                       0     1
TimeBin                              
(86399.999, 1361004.4]    57423  1631
(1361004.4, 2310159.6]    57859  1195
(2310159.6, 3864163.9]    56854  2200
(3864163.9, 5592303.6]    56522  2532
(5592303.6, 7306527.5]    56718  2336
(7306527.5, 8745782.4]    56960  2094
(8745782.4, 10437998.1]   56504  2550
(10437998.1, 12192853.6]  56993  2061
(12192853.6, 13990907.7]  57203  1851
(13990907.7, 15811131.0]  56841  2213


In [7]:
# Reproducible temporal stratified sampling
np.random.seed(42)

fraud_parts = []
legit_parts = []

for time_bin in ieee["TimeBin"].cat.categories:

    bin_data = ieee[ieee["TimeBin"] == time_bin]

    fraud = bin_data[bin_data["isFraud"] == 1].sample(
        n=100,
        random_state=42
    )

    legitimate = bin_data[bin_data["isFraud"] == 0].sample(
        n=100,
        random_state=42
    )

    fraud_parts.append(fraud)
    legit_parts.append(legitimate)

# Combine the samples
ieee_selected = pd.concat(
    fraud_parts + legit_parts,
    ignore_index=True
)

# Shuffle the final dataset
ieee_selected = ieee_selected.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Remove temporary bin column
ieee_selected = ieee_selected.drop(columns=["TimeBin"])

print("Shape:", ieee_selected.shape)

print("\nClass distribution:")
print(ieee_selected["isFraud"].value_counts())

Shape: (2000, 2)

Class distribution:
isFraud
0    1000
1    1000
Name: count, dtype: int64


In [8]:
print("Time statistics by class:")
print(
    ieee_selected.groupby("isFraud")["TransactionDT"]
    .agg(["count", "min", "median", "mean", "max"])
)

print("\nTime–FraudLabel correlation:")
print(
    ieee_selected["TransactionDT"].corr(
        ieee_selected["isFraud"]
    )
)

Time statistics by class:
         count     min     median         mean       max
isFraud                                                 
0         1000   90805  7308583.0  7368119.527  15809269
1         1000  126638  7301412.0  7366676.832  15807005

Time–FraudLabel correlation:
-0.00015601501684343178


In [9]:
# Recreate the same 10 temporal bins for validation
ieee_selected["TimeBin"] = pd.qcut(
    ieee_selected["TransactionDT"],
    q=10,
    duplicates="drop"
)

print(
    pd.crosstab(
        ieee_selected["TimeBin"],
        ieee_selected["isFraud"]
    )
)

isFraud                     0    1
TimeBin                           
(90804.999, 1367477.3]    100  100
(1367477.3, 2316084.6]    100  100
(2316084.6, 3875497.5]    100  100
(3875497.5, 5593740.4]    100  100
(5593740.4, 7308583.0]    100  100
(7308583.0, 8747821.4]    100  100
(8747821.4, 10437648.9]   100  100
(10437648.9, 12192880.8]  100  100
(12192880.8, 13989897.8]  100  100
(13989897.8, 15809269.0]  100  100


In [10]:
# Remove temporary validation column
ieee_selected = ieee_selected.drop(columns=["TimeBin"])

# Save corrected IEEE-CIS dataset
ieee_selected.to_csv(
    "IEEE_selected_2000_temporally_matched.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", ieee_selected.shape)
print("File: IEEE_selected_2000_temporally_matched.csv")

Saved successfully.
Shape: (2000, 2)
File: IEEE_selected_2000_temporally_matched.csv


In [11]:
import os

print(os.listdir("ulb_data"))

['creditcard.csv']


In [12]:
ulb = pd.read_csv(
    "ulb_data/creditcard.csv",
    usecols=["Time", "Amount", "Class"]
)

print("Shape:", ulb.shape)

print("\nClass distribution:")
print(ulb["Class"].value_counts())

print("\nTime statistics by class:")
print(
    ulb.groupby("Class")["Time"]
    .agg(["count", "min", "median", "mean", "max"])
)

Shape: (284807, 3)

Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Time statistics by class:
        count    min   median          mean       max
Class                                                
0      284315    0.0  84711.0  94838.202258  172792.0
1         492  406.0  75568.5  80746.806911  170348.0


In [13]:
# Divide the ULB timeline into 10 temporal bins
ulb["TimeBin"] = pd.qcut(
    ulb["Time"],
    q=10,
    duplicates="drop"
)

ulb_time_distribution = pd.crosstab(
    ulb["TimeBin"],
    ulb["Class"]
)

print(ulb_time_distribution)

Class                     0   1
TimeBin                        
(-0.001, 35027.0]     28389  93
(35027.0, 47694.2]    28416  64
(47694.2, 60776.0]    28436  47
(60776.0, 73261.4]    28441  37
(73261.4, 84692.0]    28453  28
(84692.0, 120396.0]   28393  91
(120396.0, 132929.0]  28456  24
(132929.0, 145247.8]  28444  33
(145247.8, 157640.4]  28428  53
(157640.4, 172792.0]  28459  22


In [14]:
# Divide the ULB timeline into 10 temporal bins
ulb["TimeBin"] = pd.qcut(
    ulb["Time"],
    q=10,
    duplicates="drop"
)

ulb_time_distribution = pd.crosstab(
    ulb["TimeBin"],
    ulb["Class"]
)

print(ulb_time_distribution)

Class                     0   1
TimeBin                        
(-0.001, 35027.0]     28389  93
(35027.0, 47694.2]    28416  64
(47694.2, 60776.0]    28436  47
(60776.0, 73261.4]    28441  37
(73261.4, 84692.0]    28453  28
(84692.0, 120396.0]   28393  91
(120396.0, 132929.0]  28456  24
(132929.0, 145247.8]  28444  33
(145247.8, 157640.4]  28428  53
(157640.4, 172792.0]  28459  22


In [15]:
# Keep ALL 492 fraud transactions
ulb_fraud = ulb[ulb["Class"] == 1].copy()

# Select legitimate transactions matching the fraud count
# in each temporal bin
legitimate_parts = []

for time_bin in ulb["TimeBin"].cat.categories:

    bin_data = ulb[ulb["TimeBin"] == time_bin]

    fraud_count = len(
        bin_data[bin_data["Class"] == 1]
    )

    legitimate = bin_data[
        bin_data["Class"] == 0
    ].sample(
        n=fraud_count,
        random_state=42
    )

    legitimate_parts.append(legitimate)

ulb_legitimate = pd.concat(
    legitimate_parts,
    ignore_index=True
)

# Combine fraud + matched legitimate
ulb_selected = pd.concat(
    [ulb_fraud, ulb_legitimate],
    ignore_index=True
)

# Shuffle
ulb_selected = ulb_selected.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Shape:", ulb_selected.shape)

print("\nClass distribution:")
print(ulb_selected["Class"].value_counts())

Shape: (984, 4)

Class distribution:
Class
0    492
1    492
Name: count, dtype: int64


In [16]:
print("Time statistics by class:")
print(
    ulb_selected.groupby("Class")["Time"]
    .agg(["count", "min", "median", "mean", "max"])
)

print("\nTime–FraudLabel correlation:")
print(
    ulb_selected["Time"].corr(
        ulb_selected["Class"]
    )
)

Time statistics by class:
       count    min   median          mean       max
Class                                               
0        492   64.0  76206.5  82978.536585  172241.0
1        492  406.0  75568.5  80746.806911  170348.0

Time–FraudLabel correlation:
-0.023236091019930936


In [17]:
print(
    pd.crosstab(
        ulb_selected["TimeBin"],
        ulb_selected["Class"]
    )
)

Class                  0   1
TimeBin                     
(-0.001, 35027.0]     93  93
(35027.0, 47694.2]    64  64
(47694.2, 60776.0]    47  47
(60776.0, 73261.4]    37  37
(73261.4, 84692.0]    28  28
(84692.0, 120396.0]   91  91
(120396.0, 132929.0]  24  24
(132929.0, 145247.8]  33  33
(145247.8, 157640.4]  53  53
(157640.4, 172792.0]  22  22


In [18]:
# Remove temporary validation column
ulb_selected = ulb_selected.drop(columns=["TimeBin"])

# Save corrected ULB dataset
ulb_selected.to_csv(
    "ULB_selected_984_temporally_matched.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", ulb_selected.shape)
print("File: ULB_selected_984_temporally_matched.csv")

Saved successfully.
Shape: (984, 3)
File: ULB_selected_984_temporally_matched.csv


In [19]:
import os

print(os.listdir("paysim_data"))

['PS_20174392719_1491204439457_log.csv']


In [20]:
paysim = pd.read_csv(
    "paysim_data/PS_20174392719_1491204439457_log.csv",
    usecols=["step", "amount", "isFraud"]
)

print("Shape:", paysim.shape)

print("\nClass distribution:")
print(paysim["isFraud"].value_counts())

print("\nTime statistics by class:")
print(
    paysim.groupby("isFraud")["step"]
    .agg(["count", "min", "median", "mean", "max"])
)

Shape: (6362620, 3)

Class distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Time statistics by class:
           count  min  median        mean  max
isFraud                                       
0        6354407    1   239.0  243.235663  718
1           8213    1   367.0  368.413856  743


In [21]:
# Divide PaySim timeline into 10 temporal bins
paysim["TimeBin"] = pd.qcut(
    paysim["step"],
    q=10,
    duplicates="drop"
)

paysim_time_distribution = pd.crosstab(
    paysim["TimeBin"],
    paysim["isFraud"]
)

print(paysim_time_distribution)

isFraud              0     1
TimeBin                     
(0.999, 35.0]   659653   398
(35.0, 135.0]   631256  1148
(135.0, 166.0]  625605   350
(166.0, 206.0]  649632   410
(206.0, 239.0]  617641   399
(239.0, 281.0]  633619   488
(281.0, 323.0]  642538   450
(323.0, 355.0]  649977   320
(355.0, 399.0]  641738   498
(399.0, 743.0]  602748  3752


In [22]:
# Target number of fraud records
target_fraud = 1008

# Count fraud transactions in each temporal bin
fraud_counts = (
    paysim[paysim["isFraud"] == 1]
    .groupby("TimeBin", observed=True)
    .size()
)

# Calculate the proportional allocation
exact_allocation = fraud_counts / fraud_counts.sum() * target_fraud

# Start with the floor of each allocation
fraud_allocation = np.floor(exact_allocation).astype(int)

# Number of records still needed after flooring
remaining = target_fraud - fraud_allocation.sum()

# Give remaining records to bins with the largest fractional parts
fractional_parts = exact_allocation - fraud_allocation

bins_to_add = fractional_parts.sort_values(
    ascending=False
).index[:remaining]

fraud_allocation.loc[bins_to_add] += 1

print("Fraud allocation by time bin:")
print(fraud_allocation)

print("\nTotal fraud records:", fraud_allocation.sum())

Fraud allocation by time bin:
TimeBin
(0.999, 35.0]      49
(35.0, 135.0]     141
(135.0, 166.0]     43
(166.0, 206.0]     50
(206.0, 239.0]     49
(239.0, 281.0]     60
(281.0, 323.0]     55
(323.0, 355.0]     39
(355.0, 399.0]     61
(399.0, 743.0]    461
dtype: int64

Total fraud records: 1008


In [23]:
# Select the allocated number of fraud transactions from each time bin
fraud_parts = []

for time_bin, n in fraud_allocation.items():
    bin_fraud = paysim[
        (paysim["TimeBin"] == time_bin) &
        (paysim["isFraud"] == 1)
    ]

    selected_fraud = bin_fraud.sample(
        n=n,
        random_state=42
    )

    fraud_parts.append(selected_fraud)

paysim_fraud_selected = pd.concat(
    fraud_parts,
    ignore_index=True
)

# Now select the SAME number of legitimate transactions
# from each corresponding time bin
legitimate_parts = []

for time_bin, n in fraud_allocation.items():
    bin_legitimate = paysim[
        (paysim["TimeBin"] == time_bin) &
        (paysim["isFraud"] == 0)
    ]

    selected_legitimate = bin_legitimate.sample(
        n=n,
        random_state=42
    )

    legitimate_parts.append(selected_legitimate)

paysim_legitimate_selected = pd.concat(
    legitimate_parts,
    ignore_index=True
)

# Combine both classes
paysim_selected = pd.concat(
    [
        paysim_fraud_selected,
        paysim_legitimate_selected
    ],
    ignore_index=True
)

# Shuffle final dataset
paysim_selected = paysim_selected.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Shape:", paysim_selected.shape)

print("\nClass distribution:")
print(paysim_selected["isFraud"].value_counts())

Shape: (2016, 4)

Class distribution:
isFraud
0    1008
1    1008
Name: count, dtype: int64


In [24]:
print("Time statistics by class:")
print(
    paysim_selected.groupby("isFraud")["step"]
    .agg(["count", "min", "median", "mean", "max"])
)

print("\nTime–FraudLabel correlation:")
print(
    paysim_selected["step"].corr(
        paysim_selected["isFraud"]
    )
)

Time statistics by class:
         count  min  median        mean  max
isFraud                                     
0         1008    9   372.0  339.010913  718
1         1008    1   363.5  365.311508  743

Time–FraudLabel correlation:
0.06427065925318116


In [25]:
print(
    pd.crosstab(
        paysim_selected["TimeBin"],
        paysim_selected["isFraud"]
    )
)

isFraud           0    1
TimeBin                 
(0.999, 35.0]    49   49
(35.0, 135.0]   141  141
(135.0, 166.0]   43   43
(166.0, 206.0]   50   50
(206.0, 239.0]   49   49
(239.0, 281.0]   60   60
(281.0, 323.0]   55   55
(323.0, 355.0]   39   39
(355.0, 399.0]   61   61
(399.0, 743.0]  461  461


In [26]:
# Remove temporary validation column
paysim_selected = paysim_selected.drop(columns=["TimeBin"])

# Save corrected PaySim dataset
paysim_selected.to_csv(
    "PaySim_selected_2016_temporally_matched.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", paysim_selected.shape)
print("File: PaySim_selected_2016_temporally_matched.csv")

Saved successfully.
Shape: (2016, 3)
File: PaySim_selected_2016_temporally_matched.csv


In [27]:
# Load the three corrected temporally matched datasets

ieee_final = pd.read_csv(
    "IEEE_selected_2000_temporally_matched.csv"
)

ulb_final = pd.read_csv(
    "ULB_selected_984_temporally_matched.csv"
)

paysim_final = pd.read_csv(
    "PaySim_selected_2016_temporally_matched.csv"
)

print("IEEE-CIS:", ieee_final.shape)
print("ULB:", ulb_final.shape)
print("PaySim:", paysim_final.shape)

IEEE-CIS: (2000, 2)
ULB: (984, 3)
PaySim: (2016, 3)


In [28]:
print("IEEE-CIS columns:")
print(ieee_final.columns.tolist())

print("\nULB columns:")
print(ulb_final.columns.tolist())

print("\nPaySim columns:")
print(paysim_final.columns.tolist())

IEEE-CIS columns:
['isFraud', 'TransactionDT']

ULB columns:
['Time', 'Amount', 'Class']

PaySim columns:
['step', 'amount', 'isFraud']


In [29]:
["TransactionID", "TransactionDT", "ProductCD", "isFraud"]

['TransactionID', 'TransactionDT', 'ProductCD', 'isFraud']

In [30]:
# Load the required IEEE-CIS fields from the ORIGINAL Kaggle dataset

ieee_raw = pd.read_csv(
    "ieee_data/train_transaction.csv",
    usecols=[
        "TransactionID",
        "TransactionDT",
        "ProductCD",
        "isFraud"
    ]
)

print("Shape:", ieee_raw.shape)

print("\nClass distribution:")
print(ieee_raw["isFraud"].value_counts())

print("\nProductCD distribution:")
print(ieee_raw["ProductCD"].value_counts())

print("\nColumns:")
print(ieee_raw.columns.tolist())

Shape: (590540, 4)

Class distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64

ProductCD distribution:
ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64

Columns:
['TransactionID', 'isFraud', 'TransactionDT', 'ProductCD']


In [31]:
# Create 10 temporal bins on the ORIGINAL IEEE-CIS data
ieee_raw["TimeBin"] = pd.qcut(
    ieee_raw["TransactionDT"],
    q=10,
    duplicates="drop"
)

# Select 100 fraud + 100 legitimate from each temporal bin
fraud_parts = []
legit_parts = []

for time_bin in ieee_raw["TimeBin"].cat.categories:

    bin_data = ieee_raw[
        ieee_raw["TimeBin"] == time_bin
    ]

    fraud = bin_data[
        bin_data["isFraud"] == 1
    ].sample(
        n=100,
        random_state=42
    )

    legitimate = bin_data[
        bin_data["isFraud"] == 0
    ].sample(
        n=100,
        random_state=42
    )

    fraud_parts.append(fraud)
    legit_parts.append(legitimate)

# Combine
ieee_selected_full = pd.concat(
    fraud_parts + legit_parts,
    ignore_index=True
)

# Shuffle
ieee_selected_full = ieee_selected_full.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Shape:", ieee_selected_full.shape)

print("\nClass distribution:")
print(
    ieee_selected_full["isFraud"].value_counts()
)

print("\nProductCD distribution:")
print(
    ieee_selected_full["ProductCD"].value_counts()
)

Shape: (2000, 5)

Class distribution:
isFraud
0    1000
1    1000
Name: count, dtype: int64

ProductCD distribution:
ProductCD
W    1172
C     511
R     133
H     126
S      58
Name: count, dtype: int64


In [32]:
print("Time statistics by class:")
print(
    ieee_selected_full.groupby("isFraud")["TransactionDT"]
    .agg(["count", "min", "median", "mean", "max"])
)

print("\nTime–FraudLabel correlation:")
print(
    ieee_selected_full["TransactionDT"].corr(
        ieee_selected_full["isFraud"]
    )
)

print("\nTemporal-bin validation:")
print(
    pd.crosstab(
        ieee_selected_full["TimeBin"],
        ieee_selected_full["isFraud"]
    )
)

Time statistics by class:
         count     min     median         mean       max
isFraud                                                 
0         1000   90805  7308583.0  7368119.527  15809269
1         1000  126638  7301412.0  7366676.832  15807005

Time–FraudLabel correlation:
-0.00015601501684343178

Temporal-bin validation:
isFraud                     0    1
TimeBin                           
(86399.999, 1361004.4]    100  100
(1361004.4, 2310159.6]    100  100
(2310159.6, 3864163.9]    100  100
(3864163.9, 5592303.6]    100  100
(5592303.6, 7306527.5]    100  100
(7306527.5, 8745782.4]    100  100
(8745782.4, 10437998.1]   100  100
(10437998.1, 12192853.6]  100  100
(12192853.6, 13990907.7]  100  100
(13990907.7, 15811131.0]  100  100


In [33]:
# Remove temporary validation column
ieee_selected_full = ieee_selected_full.drop(
    columns=["TimeBin"]
)

# Save corrected IEEE-CIS source dataset
ieee_selected_full.to_csv(
    "IEEE_selected_2000_temporally_matched_with_type.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", ieee_selected_full.shape)
print("Columns:", ieee_selected_full.columns.tolist())
print("File: IEEE_selected_2000_temporally_matched_with_type.csv")

Saved successfully.
Shape: (2000, 4)
Columns: ['TransactionID', 'isFraud', 'TransactionDT', 'ProductCD']
File: IEEE_selected_2000_temporally_matched_with_type.csv


In [34]:
# Load required PaySim fields from the ORIGINAL Kaggle dataset

paysim_raw = pd.read_csv(
    "paysim_data/PS_20174392719_1491204439457_log.csv",
    usecols=[
        "step",
        "amount",
        "type",
        "isFraud"
    ]
)

print("Shape:", paysim_raw.shape)

print("\nClass distribution:")
print(paysim_raw["isFraud"].value_counts())

print("\nTransaction type distribution:")
print(paysim_raw["type"].value_counts())

print("\nColumns:")
print(paysim_raw.columns.tolist())

Shape: (6362620, 4)

Class distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Transaction type distribution:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

Columns:
['step', 'type', 'amount', 'isFraud']


In [35]:
# Create the same 10 temporal bins on the raw PaySim data
paysim_raw["TimeBin"] = pd.qcut(
    paysim_raw["step"],
    q=10,
    duplicates="drop"
)

# Original fraud counts in each bin
fraud_counts = (
    paysim_raw[paysim_raw["isFraud"] == 1]
    .groupby("TimeBin", observed=True)
    .size()
)

# Target: exactly 1,008 fraud records
target_fraud = 1008

# Proportional allocation
exact_allocation = (
    fraud_counts / fraud_counts.sum()
) * target_fraud

fraud_allocation = np.floor(
    exact_allocation
).astype(int)

remaining = (
    target_fraud - fraud_allocation.sum()
)

fractional_parts = (
    exact_allocation - fraud_allocation
)

bins_to_add = fractional_parts.sort_values(
    ascending=False
).index[:remaining]

fraud_allocation.loc[bins_to_add] += 1

print("Fraud allocation:")
print(fraud_allocation)

print("\nTotal:", fraud_allocation.sum())

Fraud allocation:
TimeBin
(0.999, 35.0]      49
(35.0, 135.0]     141
(135.0, 166.0]     43
(166.0, 206.0]     50
(206.0, 239.0]     49
(239.0, 281.0]     60
(281.0, 323.0]     55
(323.0, 355.0]     39
(355.0, 399.0]     61
(399.0, 743.0]    461
dtype: int64

Total: 1008


In [36]:
# Select fraud records according to the temporal allocation
fraud_parts = []

for time_bin, n in fraud_allocation.items():

    bin_fraud = paysim_raw[
        (paysim_raw["TimeBin"] == time_bin) &
        (paysim_raw["isFraud"] == 1)
    ]

    selected_fraud = bin_fraud.sample(
        n=n,
        random_state=42
    )

    fraud_parts.append(selected_fraud)

paysim_fraud_selected = pd.concat(
    fraud_parts,
    ignore_index=True
)

# Select the same number of legitimate records
# from the same temporal bins
legitimate_parts = []

for time_bin, n in fraud_allocation.items():

    bin_legitimate = paysim_raw[
        (paysim_raw["TimeBin"] == time_bin) &
        (paysim_raw["isFraud"] == 0)
    ]

    selected_legitimate = bin_legitimate.sample(
        n=n,
        random_state=42
    )

    legitimate_parts.append(selected_legitimate)

paysim_legitimate_selected = pd.concat(
    legitimate_parts,
    ignore_index=True
)

# Combine both classes
paysim_selected_full = pd.concat(
    [
        paysim_fraud_selected,
        paysim_legitimate_selected
    ],
    ignore_index=True
)

# Shuffle
paysim_selected_full = paysim_selected_full.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Shape:", paysim_selected_full.shape)

print("\nClass distribution:")
print(
    paysim_selected_full["isFraud"].value_counts()
)

print("\nTransaction type distribution:")
print(
    paysim_selected_full["type"].value_counts()
)

Shape: (2016, 5)

Class distribution:
isFraud
0    1008
1    1008
Name: count, dtype: int64

Transaction type distribution:
type
CASH_OUT    837
TRANSFER    586
PAYMENT     361
CASH_IN     228
DEBIT         4
Name: count, dtype: int64


In [37]:
print("Time statistics by class:")
print(
    paysim_selected_full.groupby("isFraud")["step"]
    .agg(["count", "min", "median", "mean", "max"])
)

print("\nTime–FraudLabel correlation:")
print(
    paysim_selected_full["step"].corr(
        paysim_selected_full["isFraud"]
    )
)

print("\nTemporal-bin validation:")
print(
    pd.crosstab(
        paysim_selected_full["TimeBin"],
        paysim_selected_full["isFraud"]
    )
)

Time statistics by class:
         count  min  median        mean  max
isFraud                                     
0         1008    9   372.0  339.010913  718
1         1008    1   363.5  365.311508  743

Time–FraudLabel correlation:
0.06427065925318116

Temporal-bin validation:
isFraud           0    1
TimeBin                 
(0.999, 35.0]    49   49
(35.0, 135.0]   141  141
(135.0, 166.0]   43   43
(166.0, 206.0]   50   50
(206.0, 239.0]   49   49
(239.0, 281.0]   60   60
(281.0, 323.0]   55   55
(323.0, 355.0]   39   39
(355.0, 399.0]   61   61
(399.0, 743.0]  461  461


In [38]:
# Remove temporary validation column
paysim_selected_full = paysim_selected_full.drop(
    columns=["TimeBin"]
)

# Save final PaySim source dataset with transaction type
paysim_selected_full.to_csv(
    "PaySim_selected_2016_temporally_matched_with_type.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", paysim_selected_full.shape)
print("Columns:", paysim_selected_full.columns.tolist())
print(
    "File: PaySim_selected_2016_temporally_matched_with_type.csv"
)

Saved successfully.
Shape: (2016, 4)
Columns: ['step', 'type', 'amount', 'isFraud']
File: PaySim_selected_2016_temporally_matched_with_type.csv


In [39]:
# Load the required IEEE-CIS fields from the ORIGINAL Kaggle dataset

ieee_raw_amount = pd.read_csv(
    "ieee_data/train_transaction.csv",
    usecols=[
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        "ProductCD",
        "isFraud"
    ]
)

print("Shape:", ieee_raw_amount.shape)

print("\nColumns:")
print(ieee_raw_amount.columns.tolist())

print("\nClass distribution:")
print(ieee_raw_amount["isFraud"].value_counts())

print("\nAmount statistics:")
print(
    ieee_raw_amount.groupby("isFraud")["TransactionAmt"]
    .agg(["count", "min", "median", "mean", "max"])
)

Shape: (590540, 5)

Columns:
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD']

Class distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64

Amount statistics:
          count    min  median        mean        max
isFraud                                              
0        569877  0.251    68.5  134.511665  31937.391
1         20663  0.292    75.0  149.244779   5191.000


In [40]:
# Create the same 10 temporal bins
ieee_raw_amount["TimeBin"] = pd.qcut(
    ieee_raw_amount["TransactionDT"],
    q=10,
    duplicates="drop"
)

fraud_parts = []
legit_parts = []

for time_bin in ieee_raw_amount["TimeBin"].cat.categories:

    bin_data = ieee_raw_amount[
        ieee_raw_amount["TimeBin"] == time_bin
    ]

    fraud = bin_data[
        bin_data["isFraud"] == 1
    ].sample(
        n=100,
        random_state=42
    )

    legitimate = bin_data[
        bin_data["isFraud"] == 0
    ].sample(
        n=100,
        random_state=42
    )

    fraud_parts.append(fraud)
    legit_parts.append(legitimate)

# Combine
ieee_selected_final = pd.concat(
    fraud_parts + legit_parts,
    ignore_index=True
)

# Shuffle
ieee_selected_final = ieee_selected_final.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Shape:", ieee_selected_final.shape)

print("\nClass distribution:")
print(
    ieee_selected_final["isFraud"].value_counts()
)

print("\nColumns:")
print(ieee_selected_final.columns.tolist())

Shape: (2000, 6)

Class distribution:
isFraud
0    1000
1    1000
Name: count, dtype: int64

Columns:
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'TimeBin']


In [41]:
print("Missing values:")
print(ieee_selected_final.isna().sum())

print("\nTime statistics by class:")
print(
    ieee_selected_final.groupby("isFraud")["TransactionDT"]
    .agg(["count", "min", "median", "mean", "max"])
)

print("\nTime–FraudLabel correlation:")
print(
    ieee_selected_final["TransactionDT"].corr(
        ieee_selected_final["isFraud"]
    )
)

print("\nTemporal-bin validation:")
print(
    pd.crosstab(
        ieee_selected_final["TimeBin"],
        ieee_selected_final["isFraud"]
    )
)

Missing values:
TransactionID     0
isFraud           0
TransactionDT     0
TransactionAmt    0
ProductCD         0
TimeBin           0
dtype: int64

Time statistics by class:
         count     min     median         mean       max
isFraud                                                 
0         1000   90805  7308583.0  7368119.527  15809269
1         1000  126638  7301412.0  7366676.832  15807005

Time–FraudLabel correlation:
-0.00015601501684343178

Temporal-bin validation:
isFraud                     0    1
TimeBin                           
(86399.999, 1361004.4]    100  100
(1361004.4, 2310159.6]    100  100
(2310159.6, 3864163.9]    100  100
(3864163.9, 5592303.6]    100  100
(5592303.6, 7306527.5]    100  100
(7306527.5, 8745782.4]    100  100
(8745782.4, 10437998.1]   100  100
(10437998.1, 12192853.6]  100  100
(12192853.6, 13990907.7]  100  100
(13990907.7, 15811131.0]  100  100


In [42]:
# Remove temporary validation column
ieee_selected_final = ieee_selected_final.drop(
    columns=["TimeBin"]
)

# Save final IEEE-CIS source dataset
ieee_selected_final.to_csv(
    "IEEE_selected_2000_final.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", ieee_selected_final.shape)
print("Columns:", ieee_selected_final.columns.tolist())
print("File: IEEE_selected_2000_final.csv")

Saved successfully.
Shape: (2000, 5)
Columns: ['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD']
File: IEEE_selected_2000_final.csv


In [43]:
print("IEEE-CIS:")
print(ieee_selected_final.columns.tolist())

print("\nULB:")
print(ulb_final.columns.tolist())

print("\nPaySim:")
print(paysim_selected_full.columns.tolist())

IEEE-CIS:
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD']

ULB:
['Time', 'Amount', 'Class']

PaySim:
['step', 'type', 'amount', 'isFraud']


In [44]:
# ============================================
# STEP 32: CREATE 5,000-RECORD HARMONIZED DATASET
# ============================================

# ---------- IEEE-CIS ----------
ieee_harmonized = pd.DataFrame({
    "Time": ieee_selected_final["TransactionDT"],
    "Amount": ieee_selected_final["TransactionAmt"],
    "TransactionType": ieee_selected_final["ProductCD"],
    "FraudLabel": ieee_selected_final["isFraud"],
    "Source": "IEEE-CIS"
})


# ---------- ULB ----------
ulb_harmonized = pd.DataFrame({
    "Time": ulb_final["Time"],
    "Amount": ulb_final["Amount"],
    "TransactionType": "CardTransaction",
    "FraudLabel": ulb_final["Class"],
    "Source": "ULB"
})


# ---------- PaySim ----------
paysim_harmonized = pd.DataFrame({
    "Time": paysim_selected_full["step"],
    "Amount": paysim_selected_full["amount"],
    "TransactionType": paysim_selected_full["type"],
    "FraudLabel": paysim_selected_full["isFraud"],
    "Source": "PaySim"
})


# ---------- Combine ----------
harmonized_5000 = pd.concat(
    [
        ieee_harmonized,
        ulb_harmonized,
        paysim_harmonized
    ],
    ignore_index=True
)


print("Shape:", harmonized_5000.shape)

print("\nColumns:")
print(harmonized_5000.columns.tolist())

print("\nClass distribution:")
print(harmonized_5000["FraudLabel"].value_counts())

print("\nSource distribution:")
print(harmonized_5000["Source"].value_counts())

print("\nTransactionType distribution:")
print(harmonized_5000["TransactionType"].value_counts())

Shape: (5000, 5)

Columns:
['Time', 'Amount', 'TransactionType', 'FraudLabel', 'Source']

Class distribution:
FraudLabel
0    2500
1    2500
Name: count, dtype: int64

Source distribution:
Source
PaySim      2016
IEEE-CIS    2000
ULB          984
Name: count, dtype: int64

TransactionType distribution:
TransactionType
W                  1172
CardTransaction     984
CASH_OUT            837
TRANSFER            586
C                   511
PAYMENT             361
CASH_IN             228
R                   133
H                   126
S                    58
DEBIT                 4
Name: count, dtype: int64


In [45]:
print("===== MISSING VALUES =====")
print(harmonized_5000.isna().sum())

print("\n===== DUPLICATE ROWS =====")
print(
    "Duplicate rows:",
    harmonized_5000.duplicated().sum()
)

print("\n===== DATA TYPES =====")
print(harmonized_5000.dtypes)

print("\n===== AMOUNT CHECK =====")
print(
    harmonized_5000["Amount"].describe()
)

print(
    "\nNegative amounts:",
    (harmonized_5000["Amount"] < 0).sum()
)

print("\n===== FRAUD BY SOURCE =====")
print(
    pd.crosstab(
        harmonized_5000["Source"],
        harmonized_5000["FraudLabel"]
    )
)

print("\n===== TRANSACTION TYPE BY SOURCE =====")
print(
    pd.crosstab(
        harmonized_5000["Source"],
        harmonized_5000["TransactionType"]
    )
)

===== MISSING VALUES =====
Time               0
Amount             0
TransactionType    0
FraudLabel         0
Source             0
dtype: int64

===== DUPLICATE ROWS =====
Duplicate rows: 21

===== DATA TYPES =====
Time               float64
Amount             float64
TransactionType     object
FraudLabel           int64
Source              object
dtype: object

===== AMOUNT CHECK =====
count    5.000000e+03
mean     3.067072e+05
std      1.151264e+06
min      0.000000e+00
25%      4.641700e+01
50%      2.060000e+02
75%      8.901432e+04
max      1.000000e+07
Name: Amount, dtype: float64

Negative amounts: 0

===== FRAUD BY SOURCE =====
FraudLabel     0     1
Source                
IEEE-CIS    1000  1000
PaySim      1008  1008
ULB          492   492

===== TRANSACTION TYPE BY SOURCE =====
TransactionType    C  CASH_IN  CASH_OUT  CardTransaction  DEBIT    H  PAYMENT  \
Source                                                                          
IEEE-CIS         511        0        

In [46]:
# Identify exact duplicate harmonized rows
duplicates = harmonized_5000[
    harmonized_5000.duplicated(
        keep=False
    )
].sort_values(
    by=["Source", "Time", "Amount", "TransactionType", "FraudLabel"]
)

print("Number of rows involved in duplicates:", len(duplicates))

print("\nDuplicate rows:")
print(duplicates.to_string(index=False))

Number of rows involved in duplicates: 36

Duplicate rows:
    Time      Amount TransactionType  FraudLabel Source
   387.0 10000000.00        CASH_OUT           1 PaySim
   387.0 10000000.00        CASH_OUT           1 PaySim
   409.0 10000000.00        TRANSFER           1 PaySim
   409.0 10000000.00        TRANSFER           1 PaySim
 68207.0        1.00 CardTransaction           1    ULB
 68207.0        1.00 CardTransaction           1    ULB
 68207.0        1.00 CardTransaction           1    ULB
 68207.0        1.00 CardTransaction           1    ULB
 68207.0        1.00 CardTransaction           1    ULB
 68207.0        1.00 CardTransaction           1    ULB
 84204.0        0.00 CardTransaction           1    ULB
 84204.0        0.00 CardTransaction           1    ULB
 84204.0      512.25 CardTransaction           1    ULB
 84204.0      512.25 CardTransaction           1    ULB
 85285.0        0.00 CardTransaction           1    ULB
 85285.0        0.00 CardTransaction         

In [47]:
print("Duplicate groups by source:")

print(
    duplicates.groupby("Source").size()
)

print("\nDuplicate rows by source:")

print(
    duplicates["Source"].value_counts()
)

Duplicate groups by source:
Source
PaySim     4
ULB       32
dtype: int64

Duplicate rows by source:
Source
ULB       32
PaySim     4
Name: count, dtype: int64


In [48]:
print("===== FINAL HARMONIZED DATASET CHECK =====")

print("\nShape:")
print(harmonized_5000.shape)

print("\nMissing values:")
print(harmonized_5000.isna().sum())

print("\nFraud distribution:")
print(harmonized_5000["FraudLabel"].value_counts())

print("\nSource distribution:")
print(harmonized_5000["Source"].value_counts())

print("\nFraud distribution by source:")
print(
    pd.crosstab(
        harmonized_5000["Source"],
        harmonized_5000["FraudLabel"]
    )
)

print("\nDuplicate rows retained:")
print(harmonized_5000.duplicated().sum())

===== FINAL HARMONIZED DATASET CHECK =====

Shape:
(5000, 5)

Missing values:
Time               0
Amount             0
TransactionType    0
FraudLabel         0
Source             0
dtype: int64

Fraud distribution:
FraudLabel
0    2500
1    2500
Name: count, dtype: int64

Source distribution:
Source
PaySim      2016
IEEE-CIS    2000
ULB          984
Name: count, dtype: int64

Fraud distribution by source:
FraudLabel     0     1
Source                
IEEE-CIS    1000  1000
PaySim      1008  1008
ULB          492   492

Duplicate rows retained:
21


In [49]:
# Save the final harmonized dataset

harmonized_5000.to_csv(
    "harmonized_5000_temporally_matched.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", harmonized_5000.shape)
print("File: harmonized_5000_temporally_matched.csv")

Saved successfully.
Shape: (5000, 5)
File: harmonized_5000_temporally_matched.csv
